# Recency-five public result and existing 3/4/5 average review

This notebook audits recorded evidence. It does not fit new models or select a final submission.

In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
RESULTS = Path.cwd() / 'recency_average_review_results'
if not RESULTS.is_dir():
    RESULTS = Path('/Users/mlvlo/.codex/.chatgpt-projects/g-p-6a901a23216481918fe5444e40decc24/student_model_review/recency_average_review_results')
facts = json.loads((RESULTS / 'review_verification.json').read_text())
FROZEN = RESULTS / 'reviewed_prediction_files'
parts = [pd.read_csv(FROZEN / f'submission_lowreg_pair_recent{h}.csv') for h in (3, 4, 5)]
average = pd.read_csv(FROZEN / 'submission_lowreg_pair_recent345_avg.csv')
assert all(p['index'].equals(average['index']) for p in parts)
assert np.max(np.abs(average.target - sum(p.target for p in parts) / 3)) <= 1e-15
assert not facts['new_model_fitting'] and facts['average_public_auc'] is None
display(pd.read_csv(RESULTS / 'observed_recency_curve.csv'))
display(pd.read_csv(RESULTS / 'prediction_file_reviews.csv'))
print('Existing average verified. Current public best: 0.95471; average public score unknown.')


Existing average verified. Current public best: 0.95471; average public score unknown.


,weighting,verified_public_auc,local_15_mean,local_provenance
0,NaN,0.94701,0.955496,Previously independently calculated fixed-mask...
1,Half-life 7,0.94989,0.955913,Previously independently calculated
2,Half-life 5,0.95154,0.953800,Rounded value quoted by Claude; not independen...
3,Half-life 4,0.95471,0.950176,Previously independently calculated
4,Half-life 3,0.95270,0.947000,Rounded value quoted by Claude; not independen...


,setting,filename,sha256,rows,schema_indices_range_passed,rank_correlation_vs_half4,top1000_overlap_vs_half4
0,3,submission_lowreg_pair_recent3.csv,ae15962f184050fe9e2a7bcc6059dd5ca0bfcdce794e3e...,15329,True,0.970358,931
1,4,submission_lowreg_pair_recent4.csv,9a011de2931c31e4d641471f9e673f8a553e9cfafeeff7...,15329,True,1.000000,1000
2,5,submission_lowreg_pair_recent5.csv,78d667853450de85c2fbbdb6e909a14b08d6e43c93a992...,15329,True,0.985650,921
3,345_average,submission_lowreg_pair_recent345_avg.csv,a3556f4617da027a406d6462d1be2f1612440296f1c368...,15329,True,0.995283,964


### 41.1. Updated evidence and review of Claude's prepared average

**13 October 2026.** The new screenshot confirms **0.95154** for the half-life-five pair and **0.95270** for the half-life-three pair. The current best remains **0.95471** at half-life four. Five is **0.00317 below** four; three is **0.00201 below** four. The remaining gap to 0.96000 is **0.00529**.

| Recency half-life | Confirmed public ROC-AUC | Local mean of 15 checks | Local value provenance |
|---|---:|---:|---|
| None | 0.94701 | 0.955496 | Prior independent fixed-mask control |
| 7 | 0.94989 | 0.955913 | Prior independent comparison |
| 5 | 0.95154 | approximately 0.9538 | Quoted Claude result, not reproduced in this review |
| 4 | **0.95471** | 0.950176 | Prior independent comparison |
| 3 | 0.95270 | approximately 0.9470 | Quoted Claude result, not reproduced in this review |

The new evidence supports a bounded conclusion: recency weighting helped the tested pair compared with no weighting, and four is the best of the five tested public settings. It is reasonable to stop spending submissions on finer half-life changes for now. Four is not proved optimal beyond these candidates, and no private score is known.

### Interpreting the peak without inventing an uncertainty estimate

Claude suggested that four's score was inflated by about 0.002 and its true/private performance was near 0.953. Those numbers are not measured by the available data. The five scores concern the same public subset; hidden row labels, paired error estimates and a private evaluation are unavailable. A local maximum can reflect useful learned rules, subset-specific performance, selection bias or several effects together. It does not establish a numeric luck correction.

The argument that recency must have a smooth effect is also insufficient. Tree models choose discrete partitions, and changing weights can change which splits are selected. Non-smooth changes in public rankings need not be explained solely by chance. This is an inference about the learning mechanism, not a diagnosis of the hidden result. [XGBoost's boosted-tree tutorial](https://xgboost.readthedocs.io/en/stable/tutorials/model.html) explains the split-selection mechanism; it supplies general method background, not competition data.

The repeated chronological checks and inherited six-feature mask limitations recorded in Sections 36–40 still apply. Additional later-cutoff development checks are a useful proposed complement, but they would overlap prior development and have shorter horizons. They are not independent hidden-score estimates.

### Audit of the existing equal 3/4/5 probability average

The supplied make_recency_average.py aligns the indices and computes `(p3 + p4 + p5) / 3`. Each component is already an equal XGBoost/LightGBM pair, so the resulting prediction is an equal average of six fitted component predictions. This averages predictions after fitting; it is not equivalent to fitting one model with averaged row weights or a new intermediate half-life.

The existing `submission_lowreg_pair_recent345_avg.csv` contains **15,329** unique indices, in exactly the supplied sample/test order, with finite probabilities in [0, 1]. Its scores match the arithmetic mean of the original three CSVs within **1.110223e-16**, attributable to floating-point/CSV rounding. All three original component hashes match their earlier reviews. The average's SHA-256 is **a3556f4617da027a406d6462d1be2f1612440296f1c368a497f234134eee5621**.

The average's rank correlation with four is **0.995283** and it shares **964 of the top 1,000** predicted transactions with four. These are label-free similarity diagnostics, not accuracy estimates or evidence that their errors are independent. The probability average is correctly formed, but its public ROC-AUC remains unknown. Averaging individual ROC-AUC values does not give the AUC of averaged predictions.

Averaging is a reasonable mechanism for reducing dependence on one particular setting when model weaknesses complement each other; it can also dilute a useful component. The high similarity here means any robustness benefit is unproven and a large score increase is not supported by the available evidence. [Scikit-learn's ensemble documentation](https://scikit-learn.org/stable/modules/ensemble.html#voting-classifier) describes probability averaging as a way to combine model strengths, not a guarantee of improved accuracy.

### Proposed student decision and subsequent work

For the reported remaining submission today, the already-prepared equal average is a defensible comparison if I choose to test sensitivity to the exact recency setting. This is a modest follow-up, not a justified forecast above 0.96. Keeping the remaining attempt is also possible; no submission quota or deadline was independently queried in this review.

A score near four would show that the average preserves public ranking quality. It would not prove a safer private score. A lower public score would favour four on those public rows, without proving that four is better on all hidden rows. Keep four as the current public reference; final assessment selections remain my decision. Three is currently the next-highest scored pair. The average should not automatically replace it based on an undefined rule such as 'close to 0.954'.

For subsequent experiments, freeze half-life four as the reference and choose one small, fixed change at a time, comparing against the exact reference with later-period diagnostics as well as the existing checks. Changes to tree size, training length, regularisation, weight floors or cutoffs have not been selected or run by Codex here. A cutoff actually drops rows; a positive weight floor retains them, as the existing exponential recipe already does. The 8,770 weight-concentration effective count at four is not a literal reduction from 31,235 fitting rows. Prediction-file checks verify format and reproducibility; they do not certify future accuracy.

**AI assistance record:** Claude supplied the existing average and generation script. OpenAI Codex read the new screenshot and latest journal, checked the supplied probability arithmetic, file alignment, source and hashes, computed label-free similarity diagnostics, and drafted this factual review. No model fitting, new blend-weight search, external transaction data, automatic candidate selection or Kaggle upload occurred. The component files, prepared average, script, screenshot and review evidence are preserved in recency_average_review_results/.


![Five observed recency settings](recency_average_review_results/observed_recency_curve.png)

Local values for half-lives three and five are rounded quoted results. There is no private-score forecast or confidence interval.